## 1.计算A股流通市值和A股总市值
## Calculate the free-float market capitalization and total market capitalization of A-shares

In [4]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

capital_dt = pd.read_parquet('data/capital.parquet')
capital_dt=capital_dt.sort_values(by=['stock_code', 'change_date'], ascending=[True, True])
capital_dt.rename(columns={'change_date': 'trade_date'}, inplace=True)
price_dt = pd.read_parquet('data/eod_prices.parquet')
price_dt=price_dt.sort_values(by=['stock_code', 'trade_date'], ascending=[True, True])
mv_factor_dt=pd.merge(price_dt, capital_dt, on=['stock_code', 'trade_date'], how='left')
mv_factor_dt.fillna(method='ffill', inplace=True)
mv_factor_dt.head()


,stock_code,trade_date,close_price,adjusted_close,adj_factor,volume,turnover,total_shares,float_shares,total_a_shares,float_a_shares
0,000001.SZ,20130104,15.99,355.0546,22.204789,443851.37,717567.5466,NaN,NaN,NaN,NaN
1,000001.SZ,20130107,16.30,361.9381,22.204789,357169.25,578450.4876,NaN,NaN,NaN,NaN
2,000001.SZ,20130108,16.00,355.2766,22.204789,312479.12,501360.0937,NaN,NaN,NaN,NaN
3,000001.SZ,20130109,15.86,352.1680,22.204789,251329.15,399696.1831,NaN,NaN,NaN,NaN
4,000001.SZ,20130110,15.87,352.3900,22.204789,240030.27,383347.7326,NaN,NaN,NaN,NaN


In [5]:
mv_factor_dt['float_a_shares_mv']=mv_factor_dt['float_a_shares']*mv_factor_dt['close_price']
mv_factor_dt['total_a_shares_mv']=mv_factor_dt['total_a_shares']*mv_factor_dt['close_price']
mv_factor_dt.tail()

,stock_code,trade_date,close_price,adjusted_close,adj_factor,volume,turnover,total_shares,float_shares,total_a_shares,float_a_shares,float_a_shares_mv,total_a_shares_mv
12065642,920992.BJ,20251124,17.93,18.5230,1.033073,14658.45,26244.446,9674.163836,4740.627222,9673.609463,4739.70589,84982.926613,173447.817676
12065643,920992.BJ,20251125,18.05,18.6470,1.033073,10126.13,18280.210,9674.163836,4740.627222,9673.609463,4739.70589,85551.691320,174608.650812
12065644,920992.BJ,20251126,17.80,18.3887,1.033073,10567.74,19017.103,9674.163836,4740.627222,9673.609463,4739.70589,84366.764847,172190.248446
12065645,920992.BJ,20251127,17.65,18.2337,1.033073,7207.73,12856.675,9674.163836,4740.627222,9673.609463,4739.70589,83655.808964,170739.207026
12065646,920992.BJ,20251128,17.58,18.1614,1.033073,7501.27,13195.610,9674.163836,4740.627222,9673.609463,4739.70589,83324.029551,170062.054364


## 2.插入行业分类信息,计算市值因子
## Insert industry classification information and calculate market capitalization factors
*A股流通市值 = close_price * float_a_shares*

*A股总市值 = close_price * total_a_shares*

(PS:理论上小市值策略应该只看A股流通市值，直接跟资金盘相关)
(PS: The small-cap strategy should only consider the free-float market capitalization of A-shares, which is directly related to the money market.)

*市值因子1 = -Ln(A股流通市值) 全局标准化*
(Market capitalization factor 1: Global standardization of the free-float market capitalization of A-shares)

*市值因子2 = -Ln(A股总市值) 全局标准化*
(Market capitalization factor 2: Global standardization of the total market capitalization of A-shares)


In [6]:
cat_df=pd.read_parquet('data/ref_data/Stock_Industry_Year.parquet')
mv_factor_dt['industry_name'] = mv_factor_dt.assign(year=pd.to_datetime(mv_factor_dt['trade_date'].astype(str)).dt.year).merge(cat_df[['stock_code', 'year', 'industry_name']], on=['stock_code', 'year'], how='left')['industry_name']

In [7]:
import numpy as np
mv_factor_dt.dropna(inplace=True)
mv_factor_dt['mv_flt']=-np.log(mv_factor_dt['float_a_shares_mv'])
mv_factor_dt['mv_tot']=-np.log(mv_factor_dt['total_a_shares_mv'])
mv_factor_dt.head()

,stock_code,trade_date,close_price,adjusted_close,adj_factor,volume,turnover,total_shares,float_shares,total_a_shares,float_a_shares,float_a_shares_mv,total_a_shares_mv,industry_name,mv_flt,mv_tot
102,000001.SZ,20130614,19.06,423.2233,22.204789,328553.36,6.251254e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.918767e+06,9.765651e+06,银行,-15.593639,-16.094382
103,000001.SZ,20130617,19.24,427.2201,22.204789,309210.22,5.961902e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.974663e+06,9.857876e+06,银行,-15.603038,-16.103781
104,000001.SZ,20130618,19.73,438.1005,22.204789,389599.38,7.701559e+05,512437.850182,310502.606910,512363.631057,310533.405247,6.126824e+06,1.010893e+07,银行,-15.628187,-16.128930
105,000001.SZ,20130619,19.24,427.2201,22.204789,369977.53,7.140700e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.974663e+06,9.857876e+06,银行,-15.603038,-16.103781
106,000001.SZ,20130620,11.18,400.6981,35.840616,903066.54,1.029738e+06,819808.237690,496861.283537,819958.402672,496892.496412,5.555258e+06,9.167135e+06,银行,-15.530255,-16.031135


In [8]:
# 按交易日期分组，对当天全股票池的mv_flt列做Z-score标准化（(值-均值)/标准差）
mv_factor_dt['mv_flt_std'] = mv_factor_dt.groupby('trade_date')['mv_flt'].transform(
    lambda x: (x - x.mean()) / x.std() if x.std() != 0 else 0  # 避免标准差为0时除以0报错
)

mv_factor_dt['mv_tot_std'] = mv_factor_dt.groupby('trade_date')['mv_tot'].transform(
    lambda x: (x - x.mean()) / x.std() if x.std() != 0 else 0  # 避免标准差为0时除以0报错
)
mv_factor_dt.head()


,stock_code,trade_date,close_price,adjusted_close,adj_factor,volume,turnover,total_shares,float_shares,total_a_shares,float_a_shares,float_a_shares_mv,total_a_shares_mv,industry_name,mv_flt,mv_tot,mv_flt_std,mv_tot_std
102,000001.SZ,20130614,19.06,423.2233,22.204789,328553.36,6.251254e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.918767e+06,9.765651e+06,银行,-15.593639,-16.094382,-1.744443,-2.218485
103,000001.SZ,20130617,19.24,427.2201,22.204789,309210.22,5.961902e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.974663e+06,9.857876e+06,银行,-15.603038,-16.103781,-1.750973,-2.226354
104,000001.SZ,20130618,19.73,438.1005,22.204789,389599.38,7.701559e+05,512437.850182,310502.606910,512363.631057,310533.405247,6.126824e+06,1.010893e+07,银行,-15.628187,-16.128930,-1.774256,-2.250456
105,000001.SZ,20130619,19.24,427.2201,22.204789,369977.53,7.140700e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.974663e+06,9.857876e+06,银行,-15.603038,-16.103781,-1.757097,-2.230350
106,000001.SZ,20130620,11.18,400.6981,35.840616,903066.54,1.029738e+06,819808.237690,496861.283537,819958.402672,496892.496412,5.555258e+06,9.167135e+06,银行,-15.530255,-16.031135,-1.726456,-2.194585


In [9]:
mv_factor_dt.tail()

,stock_code,trade_date,close_price,adjusted_close,adj_factor,volume,turnover,total_shares,float_shares,total_a_shares,float_a_shares,float_a_shares_mv,total_a_shares_mv,industry_name,mv_flt,mv_tot,mv_flt_std,mv_tot_std
12065642,920992.BJ,20251124,17.93,18.5230,1.033073,14658.45,26244.446,9674.163836,4740.627222,9673.609463,4739.70589,84982.926613,173447.817676,医药生物,-11.350206,-12.063632,1.797748,1.432967
12065643,920992.BJ,20251125,18.05,18.6470,1.033073,10126.13,18280.210,9674.163836,4740.627222,9673.609463,4739.70589,85551.691320,174608.650812,医药生物,-11.356876,-12.070302,1.805374,1.440739
12065644,920992.BJ,20251126,17.80,18.3887,1.033073,10567.74,19017.103,9674.163836,4740.627222,9673.609463,4739.70589,84366.764847,172190.248446,医药生物,-11.342929,-12.056355,1.809709,1.446056
12065645,920992.BJ,20251127,17.65,18.2337,1.033073,7207.73,12856.675,9674.163836,4740.627222,9673.609463,4739.70589,83655.808964,170739.207026,医药生物,-11.334466,-12.047893,1.821782,1.458540
12065646,920992.BJ,20251128,17.58,18.1614,1.033073,7501.27,13195.610,9674.163836,4740.627222,9673.609463,4739.70589,83324.029551,170062.054364,医药生物,-11.330492,-12.043919,1.837061,1.474868


## 3.单因子(A股流通市值因子)与复合因子(市值因子简单平均赋分)

In [9]:
#简单复合
mv_factor_dt['mv_comp']=0.5*mv_factor_dt['mv_flt_std']+0.5*mv_factor_dt['mv_tot_std']
mv_factor_dt.head()

,stock_code,trade_date,close_price,adjusted_close,adj_factor,volume,turnover,total_shares,float_shares,total_a_shares,float_a_shares,float_a_shares_mv,total_a_shares_mv,industry_name,mv_flt,mv_tot,mv_flt_std,mv_tot_std,mv_comp
102,000001.SZ,20130614,19.06,423.2233,22.204789,328553.36,6.251254e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.918767e+06,9.765651e+06,银行,-15.593639,-16.094382,-1.744443,-2.218485,-1.981464
103,000001.SZ,20130617,19.24,427.2201,22.204789,309210.22,5.961902e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.974663e+06,9.857876e+06,银行,-15.603038,-16.103781,-1.750973,-2.226354,-1.988663
104,000001.SZ,20130618,19.73,438.1005,22.204789,389599.38,7.701559e+05,512437.850182,310502.606910,512363.631057,310533.405247,6.126824e+06,1.010893e+07,银行,-15.628187,-16.128930,-1.774256,-2.250456,-2.012356
105,000001.SZ,20130619,19.24,427.2201,22.204789,369977.53,7.140700e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.974663e+06,9.857876e+06,银行,-15.603038,-16.103781,-1.757097,-2.230350,-1.993723
106,000001.SZ,20130620,11.18,400.6981,35.840616,903066.54,1.029738e+06,819808.237690,496861.283537,819958.402672,496892.496412,5.555258e+06,9.167135e+06,银行,-15.530255,-16.031135,-1.726456,-2.194585,-1.960520


### 3.1 单因子(A股流通市值因子)选股、Long Only回测

In [11]:
from scipy.stats.mstats import winsorize
import pandas as pd

# ===================== 1. 数据预处理 =====================
# 处理因子数据日期格式（数字转日期）
mv_factor_dt['trade_date'] = pd.to_datetime(mv_factor_dt['trade_date'].astype(str))
mv_factor_dt=mv_factor_dt[mv_factor_dt['stock_code'].str.endswith('SZ')|mv_factor_dt['stock_code'].str.endswith('SH')]

# ===================== 2. 核心选股逻辑（全股票 分位数多空） =====================
def select_stocks_by_date(date_group):
    trade_date = date_group.name
    daily_data = date_group.copy()
    
    # 【无股票池限制】直接使用当日全部股票数据
    if len(daily_data) == 0:
        return pd.DataFrame()
    
    # 对mv_comp列做1%/99%缩尾（可选，取消注释即可启用）
    daily_data['mv_comp'] = winsorize(
        daily_data['mv_flt_std'].values, 
        limits=(0.01, 0.01),
        inclusive=(True, True)
    )
    
    # 计算全股票分位数并标记多空
    q1 = daily_data['mv_flt_std'].quantile(0.01)
    q2 = daily_data['mv_flt_std'].quantile(0.99)
    
    daily_data['signal'] = 'none'
    daily_data.loc[daily_data['mv_comp'] <= q1, 'signal'] = 'long'   # 多头
    daily_data.loc[daily_data['mv_comp'] >= q2, 'signal'] = 'short'  # 空头
    
    return daily_data

# 按交易日分组执行选股（全市场股票）
selected_stocks = mv_factor_dt.groupby('trade_date').apply(select_stocks_by_date)

# 重置索引
selected_stocks = selected_stocks.reset_index(drop=True)

# 保留核心列
final_selection_df = selected_stocks[['trade_date', 'stock_code', 
                                       'mv_comp', 'signal']]
# 保存选股结果
final_selection_df.to_parquet('records/mv_selection_v3.parquet', index=False)

In [14]:
final_selection_df.head(10)

,trade_date,stock_code,mv_comp,signal
0,2013-01-04,000002.SZ,-2.611225,long
1,2013-01-04,000004.SZ,-1.923727,none
2,2013-01-04,000005.SZ,2.073634,short
3,2013-01-04,000006.SZ,0.522409,none
4,2013-01-04,000007.SZ,-0.654657,none
5,2013-01-04,000010.SZ,-1.479479,none
6,2013-01-04,000011.SZ,0.407080,none
7,2013-01-04,000012.SZ,0.692627,none
8,2013-01-04,000014.SZ,-0.498349,none
9,2013-01-04,000016.SZ,2.073634,short


In [2]:
import warnings
from util import *

warnings.filterwarnings("ignore")

# ===================== 全多头策略配置 =====================
CONFIG = {
    "INITIAL_CASH": 1_000_000,       # 初始资金
    "START_DATE": "2016-07-01",      # 回测开始日期
    "END_DATE": "2025-06-30",        # 回测结束日期
    "REBALANCE_FREQ": 5,             # 调仓频率（每N个选股日）
    "PRICE_FILE": "data/eod_prices.parquet",    # 价格文件
    "SELECTION_FILE": "records/mv_selection_v3.parquet", # 新版选股文件
    "TRANSACTION_FEE": 0.001         # 手续费0.1%
}

# ===================== 执行全多头回测 =====================
portfolio_ser, benchmark_ser, metrics = backtest_long_only(
    initial_cash=CONFIG["INITIAL_CASH"],
    start_date=CONFIG["START_DATE"],
    end_date=CONFIG["END_DATE"],
    rebalance_freq=CONFIG["REBALANCE_FREQ"],
    price_file=CONFIG["PRICE_FILE"],
    selection_file=CONFIG["SELECTION_FILE"],
    transaction_fee_rate=CONFIG["TRANSACTION_FEE"]
)

# ===================== 输出结果 =====================
print("\n==================== 全多头策略绩效指标 ====================")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

# 绘图（净值+回撤+基准对比）
plot_results(
    portfolio_values=portfolio_ser,
    benchmark_values=benchmark_ser,
    title=f'All Long Stocks | Rebalance Every {CONFIG["REBALANCE_FREQ"]} Days | Fee 0.1%'
)

Loading data...
Number of rebalance dates: 437

=== 调仓日期：2016-07-01 ===
目标股票数：29 | 总资产：1000000.00 | 每只目标市值：34482.76
实际交易总额：1000000.00 | 手续费：1000.00 | 剩余现金：-1000.00

=== 调仓日期：2016-07-08 ===
目标股票数：29 | 总资产：1005708.30 | 每只目标市值：34679.60
实际交易总额：25775.03 | 手续费：25.78 | 剩余现金：-25.78

=== 调仓日期：2016-07-15 ===
目标股票数：29 | 总资产：1031461.88 | 每只目标市值：35567.65
实际交易总额：89798.45 | 手续费：89.80 | 剩余现金：-89.80

=== 调仓日期：2016-07-22 ===
目标股票数：29 | 总资产：1022806.32 | 每只目标市值：35269.18
实际交易总额：80156.90 | 手续费：80.16 | 剩余现金：-80.16

=== 调仓日期：2016-07-29 ===
目标股票数：29 | 总资产：1017757.60 | 每只目标市值：35095.09
实际交易总额：88241.56 | 手续费：88.24 | 剩余现金：-88.24

=== 调仓日期：2016-08-05 ===
目标股票数：29 | 总资产：1028722.17 | 每只目标市值：35473.18
实际交易总额：23664.12 | 手续费：23.66 | 剩余现金：-23.66

=== 调仓日期：2016-08-12 ===
目标股票数：29 | 总资产：1057047.58 | 每只目标市值：36449.92
实际交易总额：91988.34 | 手续费：91.99 | 剩余现金：-91.99

=== 调仓日期：2016-08-19 ===
目标股票数：29 | 总资产：1073010.38 | 每只目标市值：37000.36
实际交易总额：97843.96 | 手续费：97.84 | 剩余现金：-97.84

=== 调仓日期：2016-08-26 ===
目标股票数：29 | 总资产：1057463.57 | 每只目标市值

### 3.2 单因子 Long Short回测

In [3]:
import warnings
from util import *

warnings.filterwarnings("ignore")

# ===================== 多空对冲策略配置（独立成本参数）=====================
CONFIG = {
    "INITIAL_CASH": 1_000_000,       # 初始资金
    "START_DATE": "2016-07-01",      # 回测开始日期
    "END_DATE": "2025-06-30",        # 回测结束日期
    "REBALANCE_FREQ": 5,             # 调仓频率
    "PRICE_FILE": "data/eod_prices.parquet",
    "SELECTION_FILE": "records/mv_selection_v3.parquet",
    # 成本参数（可自由修改指定值）
    "LONG_FEE": 0.001,           # 多头手续费 0.1%
    "SHORT_TRADE_FEE": 0.000,    # 【融券专属】交易手续费 0.2%（可自定义）
    "SHORT_FINANCING_RATE": 0.00  # 融券年化利息 8%（行业常规水平）
}

# ===================== 执行多空回测 =====================
portfolio_ser, benchmark_ser, metrics = backtest_long_short(
    initial_cash=CONFIG["INITIAL_CASH"],
    start_date=CONFIG["START_DATE"],
    end_date=CONFIG["END_DATE"],
    rebalance_freq=CONFIG["REBALANCE_FREQ"],
    price_file=CONFIG["PRICE_FILE"],
    selection_file=CONFIG["SELECTION_FILE"],
    long_fee_rate=CONFIG["LONG_FEE"],
    short_trade_fee_rate=CONFIG["SHORT_TRADE_FEE"],
    short_financing_rate=CONFIG["SHORT_FINANCING_RATE"]
)

# ===================== 输出绩效 =====================
print("\n==================== 多空对冲（含融券成本）绩效指标 ====================")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

# 绘图
plot_results(
    portfolio_values=portfolio_ser,
    benchmark_values=benchmark_ser,
    title=f'Long-Short | 多头{CONFIG["LONG_FEE"]*100:.1f}% | 融券{CONFIG["SHORT_TRADE_FEE"]*100:.1f}%+{CONFIG["SHORT_FINANCING_RATE"]*100:.0f}%息',
    plot_excess=True
)

Loading data...
Number of rebalance dates: 437

==================== 多空对冲（含融券成本）绩效指标 ====================
Annualized Return: -0.0863
Volatility: 0.1351
Sharpe Ratio: -0.6393
Max Drawdown: -0.5785


### 3.3 复合因子 Long Only选股回测

In [10]:
from scipy.stats.mstats import winsorize
import pandas as pd

# ===================== 1. 数据预处理 =====================
# 处理因子数据日期格式（数字转日期）
mv_factor_dt['trade_date'] = pd.to_datetime(mv_factor_dt['trade_date'].astype(str))
mv_factor_dt=mv_factor_dt[mv_factor_dt['stock_code'].str.endswith('SZ')|mv_factor_dt['stock_code'].str.endswith('SH')]

# ===================== 2. 核心选股逻辑（全股票 分位数多空） =====================
def select_stocks_by_date(date_group):
    trade_date = date_group.name
    daily_data = date_group.copy()
    
    # 【无股票池限制】直接使用当日全部股票数据
    if len(daily_data) == 0:
        return pd.DataFrame()
    
    # 对mv_comp列做1%/99%缩尾（可选，取消注释即可启用）
    daily_data['mv_comp'] = winsorize(
        daily_data['mv_comp'].values, 
        limits=(0.01, 0.01),
        inclusive=(True, True)
    )
    
    # 计算全股票分位数并标记多空
    q1 = daily_data['mv_comp'].quantile(0.01)
    q2 = daily_data['mv_comp'].quantile(0.99)
    
    daily_data['signal'] = 'none'
    daily_data.loc[daily_data['mv_comp'] <= q1, 'signal'] = 'long'   # 多头
    daily_data.loc[daily_data['mv_comp'] >= q2, 'signal'] = 'short'  # 空头
    
    return daily_data

# 按交易日分组执行选股（全市场股票）
selected_stocks = mv_factor_dt.groupby('trade_date').apply(select_stocks_by_date)

# 重置索引
selected_stocks = selected_stocks.reset_index(drop=True)

# 保留核心列
final_selection_df = selected_stocks[['trade_date', 'stock_code', 
                                       'mv_comp', 'signal']]
# 保存选股结果
final_selection_df.to_parquet('records/mv_selection_v4.parquet', index=False)

### 3.4 策略稳健性检验（不同时间子样本）

In [16]:
import warnings
from util import *

warnings.filterwarnings("ignore")

# ===================== 全多头策略配置 =====================
CONFIG = {
    "INITIAL_CASH": 1_000_000,       # 初始资金
    "START_DATE": "2016-07-01",      # 回测开始日期
    "END_DATE": "2018-06-30",        # 回测结束日期
    "REBALANCE_FREQ": 5,             # 调仓频率（每N个选股日）
    "PRICE_FILE": "data/eod_prices.parquet",    # 价格文件
    "SELECTION_FILE": "records/mv_selection_v4.parquet", # 新版选股文件
    "TRANSACTION_FEE": 0.008         # 手续费0.1%
}

# ===================== 执行全多头回测 =====================
portfolio_ser, benchmark_ser, metrics = backtest_long_only(
    initial_cash=CONFIG["INITIAL_CASH"],
    start_date=CONFIG["START_DATE"],
    end_date=CONFIG["END_DATE"],
    rebalance_freq=CONFIG["REBALANCE_FREQ"],
    price_file=CONFIG["PRICE_FILE"],
    selection_file=CONFIG["SELECTION_FILE"],
    transaction_fee_rate=CONFIG["TRANSACTION_FEE"]
)

# ===================== 输出结果 =====================
print("\n==================== 全多头策略绩效指标 ====================")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

# 绘图（净值+回撤+基准对比）
plot_results(
    portfolio_values=portfolio_ser,
    benchmark_values=benchmark_ser,
    title=f'All Long Stocks | Rebalance Every {CONFIG["REBALANCE_FREQ"]} Days | Fee 0.1%',
    plot_excess=True
)

Loading data...
Number of rebalance dates: 98

=== 调仓日期：2016-07-01 ===
目标股票数：29 | 总资产：1000000.00 | 每只目标市值：34482.76
实际交易总额：1000000.00 | 手续费：8000.00 | 剩余现金：-8000.00

=== 调仓日期：2016-07-08 ===
目标股票数：29 | 总资产：999164.25 | 每只目标市值：34453.94
实际交易总额：28083.96 | 手续费：224.67 | 剩余现金：-224.67

=== 调仓日期：2016-07-15 ===
目标股票数：29 | 总资产：1025053.03 | 每只目标市值：35346.66
实际交易总额：19159.89 | 手续费：153.28 | 剩余现金：-153.28

=== 调仓日期：2016-07-22 ===
目标股票数：29 | 总资产：1016574.96 | 每只目标市值：35054.31
实际交易总额：9963.24 | 手续费：79.71 | 剩余现金：-79.71

=== 调仓日期：2016-07-29 ===
目标股票数：29 | 总资产：1016505.66 | 每只目标市值：35051.92
实际交易总额：91403.24 | 手续费：731.23 | 剩余现金：-731.23

=== 调仓日期：2016-08-05 ===
目标股票数：29 | 总资产：1019051.33 | 每只目标市值：35139.70
实际交易总额：80162.35 | 手续费：641.30 | 剩余现金：-641.30

=== 调仓日期：2016-08-12 ===
目标股票数：29 | 总资产：1048058.34 | 每只目标市值：36139.94
实际交易总额：92369.58 | 手续费：738.96 | 剩余现金：-738.96

=== 调仓日期：2016-08-19 ===
目标股票数：29 | 总资产：1061794.18 | 每只目标市值：36613.59
实际交易总额：96883.54 | 手续费：775.07 | 剩余现金：-775.07

=== 调仓日期：2016-08-26 ===
目标股票数：29 | 总资产：1045809.74

In [17]:
import warnings
from util import *

warnings.filterwarnings("ignore")

# ===================== 全多头策略配置 =====================
CONFIG = {
    "INITIAL_CASH": 1_000_000,       # 初始资金
    "START_DATE": "2018-07-01",      # 回测开始日期
    "END_DATE": "2020-06-30",        # 回测结束日期
    "REBALANCE_FREQ": 5,             # 调仓频率（每N个选股日）
    "PRICE_FILE": "data/eod_prices.parquet",    # 价格文件
    "SELECTION_FILE": "records/mv_selection_v4.parquet", # 新版选股文件
    "TRANSACTION_FEE": 0.008         # 手续费0.1%
}

# ===================== 执行全多头回测 =====================
portfolio_ser, benchmark_ser, metrics = backtest_long_only(
    initial_cash=CONFIG["INITIAL_CASH"],
    start_date=CONFIG["START_DATE"],
    end_date=CONFIG["END_DATE"],
    rebalance_freq=CONFIG["REBALANCE_FREQ"],
    price_file=CONFIG["PRICE_FILE"],
    selection_file=CONFIG["SELECTION_FILE"],
    transaction_fee_rate=CONFIG["TRANSACTION_FEE"]
)

# ===================== 输出结果 =====================
print("\n==================== 全多头策略绩效指标 ====================")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

# 绘图（净值+回撤+基准对比）
plot_results(
    portfolio_values=portfolio_ser,
    benchmark_values=benchmark_ser,
    title=f'All Long Stocks | Rebalance Every {CONFIG["REBALANCE_FREQ"]} Days | Fee 0.1%',
    plot_excess=True
)

Loading data...
Number of rebalance dates: 97

=== 调仓日期：2018-07-02 ===
目标股票数：35 | 总资产：1000000.00 | 每只目标市值：28571.43
实际交易总额：1000000.00 | 手续费：8000.00 | 剩余现金：-8000.00

=== 调仓日期：2018-07-09 ===
目标股票数：35 | 总资产：1015076.12 | 每只目标市值：29002.17
实际交易总额：30150.79 | 手续费：241.21 | 剩余现金：-241.21

=== 调仓日期：2018-07-16 ===
目标股票数：35 | 总资产：1015597.55 | 每只目标市值：29017.07
实际交易总额：27082.66 | 手续费：216.66 | 剩余现金：-216.66

=== 调仓日期：2018-07-23 ===
目标股票数：35 | 总资产：1030418.91 | 每只目标市值：29440.54
实际交易总额：96171.05 | 手续费：769.37 | 剩余现金：-769.37

=== 调仓日期：2018-07-30 ===
目标股票数：35 | 总资产：1034510.20 | 每只目标市值：29557.43
实际交易总额：21547.40 | 手续费：172.38 | 剩余现金：-172.38

=== 调仓日期：2018-08-06 ===
目标股票数：35 | 总资产：964743.55 | 每只目标市值：27564.10
实际交易总额：35765.42 | 手续费：286.12 | 剩余现金：-286.12

=== 调仓日期：2018-08-13 ===
目标股票数：35 | 总资产：994956.94 | 每只目标市值：28427.34
实际交易总额：140740.07 | 手续费：1125.92 | 剩余现金：-1125.92

=== 调仓日期：2018-08-20 ===
目标股票数：35 | 总资产：961674.48 | 每只目标市值：27476.41
实际交易总额：134258.14 | 手续费：1074.07 | 剩余现金：-1074.07

=== 调仓日期：2018-08-27 ===
目标股票数：35 | 总资产：100

In [18]:
import warnings
from util import *

warnings.filterwarnings("ignore")

# ===================== 全多头策略配置 =====================
CONFIG = {
    "INITIAL_CASH": 1_000_000,       # 初始资金
    "START_DATE": "2020-07-01",      # 回测开始日期
    "END_DATE": "2022-06-30",        # 回测结束日期
    "REBALANCE_FREQ": 5,             # 调仓频率（每N个选股日）
    "PRICE_FILE": "data/eod_prices.parquet",    # 价格文件
    "SELECTION_FILE": "records/mv_selection_v4.parquet", # 新版选股文件
    "TRANSACTION_FEE": 0.008         # 手续费0.1%
}

# ===================== 执行全多头回测 =====================
portfolio_ser, benchmark_ser, metrics = backtest_long_only(
    initial_cash=CONFIG["INITIAL_CASH"],
    start_date=CONFIG["START_DATE"],
    end_date=CONFIG["END_DATE"],
    rebalance_freq=CONFIG["REBALANCE_FREQ"],
    price_file=CONFIG["PRICE_FILE"],
    selection_file=CONFIG["SELECTION_FILE"],
    transaction_fee_rate=CONFIG["TRANSACTION_FEE"]
)

# ===================== 输出结果 =====================
print("\n==================== 全多头策略绩效指标 ====================")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

# 绘图（净值+回撤+基准对比）
plot_results(
    portfolio_values=portfolio_ser,
    benchmark_values=benchmark_ser,
    title=f'All Long Stocks | Rebalance Every {CONFIG["REBALANCE_FREQ"]} Days | Fee 0.1%',
    plot_excess=True
)

Loading data...
Number of rebalance dates: 98

=== 调仓日期：2020-07-01 ===
目标股票数：38 | 总资产：1000000.00 | 每只目标市值：26315.79
实际交易总额：1000000.00 | 手续费：8000.00 | 剩余现金：-8000.00

=== 调仓日期：2020-07-08 ===
目标股票数：38 | 总资产：1095418.38 | 每只目标市值：28826.80
实际交易总额：48674.10 | 手续费：389.39 | 剩余现金：-389.39

=== 调仓日期：2020-07-15 ===
目标股票数：38 | 总资产：1085303.44 | 每只目标市值：28560.62
实际交易总额：169398.40 | 手续费：1355.19 | 剩余现金：-1355.19

=== 调仓日期：2020-07-22 ===
目标股票数：38 | 总资产：1085168.00 | 每只目标市值：28557.05
实际交易总额：81296.35 | 手续费：650.37 | 剩余现金：-650.37

=== 调仓日期：2020-07-29 ===
目标股票数：38 | 总资产：1070153.14 | 每只目标市值：28161.92
实际交易总额：139173.60 | 手续费：1113.39 | 剩余现金：-1113.39

=== 调仓日期：2020-08-05 ===
目标股票数：38 | 总资产：1077294.68 | 每只目标市值：28349.86
实际交易总额：77880.78 | 手续费：623.05 | 剩余现金：-623.05

=== 调仓日期：2020-08-12 ===
目标股票数：38 | 总资产：1057479.95 | 每只目标市值：27828.42
实际交易总额：195895.34 | 手续费：1567.16 | 剩余现金：-1567.16

=== 调仓日期：2020-08-19 ===
目标股票数：38 | 总资产：1069296.76 | 每只目标市值：28139.39
实际交易总额：28871.23 | 手续费：230.97 | 剩余现金：-230.97

=== 调仓日期：2020-08-26 ===
目标股票数：38 | 总

In [19]:
import warnings
from util import *

warnings.filterwarnings("ignore")

# ===================== 全多头策略配置 =====================
CONFIG = {
    "INITIAL_CASH": 1_000_000,       # 初始资金
    "START_DATE": "2022-07-01",      # 回测开始日期
    "END_DATE": "2024-06-30",        # 回测结束日期
    "REBALANCE_FREQ": 5,             # 调仓频率（每N个选股日）
    "PRICE_FILE": "data/eod_prices.parquet",    # 价格文件
    "SELECTION_FILE": "records/mv_selection_v4.parquet", # 新版选股文件
    "TRANSACTION_FEE": 0.008         # 手续费0.1%
}

# ===================== 执行全多头回测 =====================
portfolio_ser, benchmark_ser, metrics = backtest_long_only(
    initial_cash=CONFIG["INITIAL_CASH"],
    start_date=CONFIG["START_DATE"],
    end_date=CONFIG["END_DATE"],
    rebalance_freq=CONFIG["REBALANCE_FREQ"],
    price_file=CONFIG["PRICE_FILE"],
    selection_file=CONFIG["SELECTION_FILE"],
    transaction_fee_rate=CONFIG["TRANSACTION_FEE"]
)

# ===================== 输出结果 =====================
print("\n==================== 全多头策略绩效指标 ====================")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

# 绘图（净值+回撤+基准对比）
plot_results(
    portfolio_values=portfolio_ser,
    benchmark_values=benchmark_ser,
    title=f'All Long Stocks | Rebalance Every {CONFIG["REBALANCE_FREQ"]} Days | Fee 0.1%',
    plot_excess=True
)

Loading data...
Number of rebalance dates: 97

=== 调仓日期：2022-07-01 ===
目标股票数：46 | 总资产：1000000.00 | 每只目标市值：21739.13
实际交易总额：1000000.00 | 手续费：8000.00 | 剩余现金：-8000.00

=== 调仓日期：2022-07-08 ===
目标股票数：46 | 总资产：979762.41 | 每只目标市值：21299.18
实际交易总额：111168.21 | 手续费：889.35 | 剩余现金：-889.35

=== 调仓日期：2022-07-15 ===
目标股票数：46 | 总资产：940728.88 | 每只目标市值：20450.63
实际交易总额：99139.40 | 手续费：793.12 | 剩余现金：-793.12

=== 调仓日期：2022-07-22 ===
目标股票数：46 | 总资产：941828.38 | 每只目标市值：20474.53
实际交易总额：64990.69 | 手续费：519.93 | 剩余现金：-519.93

=== 调仓日期：2022-07-29 ===
目标股票数：46 | 总资产：922889.49 | 每只目标市值：20062.82
实际交易总额：60141.62 | 手续费：481.13 | 剩余现金：-481.13

=== 调仓日期：2022-08-05 ===
目标股票数：46 | 总资产：912369.35 | 每只目标市值：19834.12
实际交易总额：58557.75 | 手续费：468.46 | 剩余现金：-468.46

=== 调仓日期：2022-08-12 ===
目标股票数：46 | 总资产：919200.13 | 每只目标市值：19982.61
实际交易总额：97477.10 | 手续费：779.82 | 剩余现金：-779.82

=== 调仓日期：2022-08-19 ===
目标股票数：46 | 总资产：908645.72 | 每只目标市值：19753.17
实际交易总额：137416.96 | 手续费：1099.34 | 剩余现金：-1099.34

=== 调仓日期：2022-08-26 ===
目标股票数：46 | 总资产：910437.64

In [20]:
import warnings
from util import *

warnings.filterwarnings("ignore")

# ===================== 全多头策略配置 =====================
CONFIG = {
    "INITIAL_CASH": 1_000_000,       # 初始资金
    "START_DATE": "2024-07-01",      # 回测开始日期
    "END_DATE": "2025-06-30",        # 回测结束日期
    "REBALANCE_FREQ": 5,             # 调仓频率（每N个选股日）
    "PRICE_FILE": "data/eod_prices.parquet",    # 价格文件
    "SELECTION_FILE": "records/mv_selection_v4.parquet", # 新版选股文件
    "TRANSACTION_FEE": 0.008         # 手续费0.1%
}

# ===================== 执行全多头回测 =====================
portfolio_ser, benchmark_ser, metrics = backtest_long_only(
    initial_cash=CONFIG["INITIAL_CASH"],
    start_date=CONFIG["START_DATE"],
    end_date=CONFIG["END_DATE"],
    rebalance_freq=CONFIG["REBALANCE_FREQ"],
    price_file=CONFIG["PRICE_FILE"],
    selection_file=CONFIG["SELECTION_FILE"],
    transaction_fee_rate=CONFIG["TRANSACTION_FEE"]
)

# ===================== 输出结果 =====================
print("\n==================== 全多头策略绩效指标 ====================")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

# 绘图（净值+回撤+基准对比）
plot_results(
    portfolio_values=portfolio_ser,
    benchmark_values=benchmark_ser,
    title=f'All Long Stocks | Rebalance Every {CONFIG["REBALANCE_FREQ"]} Days | Fee 0.1%',
    plot_excess=True
)

Loading data...
Number of rebalance dates: 49

=== 调仓日期：2024-07-01 ===
目标股票数：51 | 总资产：1000000.00 | 每只目标市值：19607.84
实际交易总额：1000000.00 | 手续费：8000.00 | 剩余现金：-8000.00

=== 调仓日期：2024-07-08 ===
目标股票数：51 | 总资产：982821.69 | 每只目标市值：19271.01
实际交易总额：24271.68 | 手续费：194.17 | 剩余现金：-194.17

=== 调仓日期：2024-07-15 ===
目标股票数：51 | 总资产：1003542.33 | 每只目标市值：19677.30
实际交易总额：59212.77 | 手续费：473.70 | 剩余现金：-473.70

=== 调仓日期：2024-07-22 ===
目标股票数：51 | 总资产：1003641.66 | 每只目标市值：19679.25
实际交易总额：101455.93 | 手续费：811.65 | 剩余现金：-811.65

=== 调仓日期：2024-07-29 ===
目标股票数：51 | 总资产：979067.37 | 每只目标市值：19197.40
实际交易总额：68868.20 | 手续费：550.95 | 剩余现金：-550.95

=== 调仓日期：2024-08-05 ===
目标股票数：51 | 总资产：959377.90 | 每只目标市值：18811.33
实际交易总额：102529.69 | 手续费：820.24 | 剩余现金：-820.24

=== 调仓日期：2024-08-12 ===
目标股票数：51 | 总资产：955498.08 | 每只目标市值：18735.26
实际交易总额：53329.57 | 手续费：426.64 | 剩余现金：-426.64

=== 调仓日期：2024-08-19 ===
目标股票数：51 | 总资产：976423.56 | 每只目标市值：19145.56
实际交易总额：64293.55 | 手续费：514.35 | 剩余现金：-514.35

=== 调仓日期：2024-08-26 ===
目标股票数：51 | 总资产：973355.97